<a href="https://colab.research.google.com/github/kalyanisingh19196-netizen/Neural-Networks-and-Deep-Learning-Lab/blob/main/Assignment-2-MLP-Backpropagation/gradient_checking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# Gradient Checking for a Neural Network

np.random.seed(42)

# Sample dataset
X = np.array([
    [0.5, 0.2],
    [0.3, 0.8],
    [0.9, 0.4]
])

y = np.array([
    [1],
    [0],
    [1]
])

# Network architecture
input_size = 2
hidden_size = 3
output_size = 1

# Initialize weights
W1 = np.random.randn(input_size, hidden_size) * 0.1
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * 0.1
b2 = np.zeros((1, output_size))

# Sigmoid activation
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Forward propagation
def forward(X, W1, b1, W2, b2):

    Z1 = X @ W1 + b1
    A1 = np.tanh(Z1)

    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    return A1, A2

# Calculate loss
def calculate_loss(X, y, W1, b1, W2, b2):

    _, A2 = forward(X, W1, b1, W2, b2)

    epsilon = 1e-8

    loss = -np.mean(
        y * np.log(A2 + epsilon) +
        (1-y) * np.log(1-A2 + epsilon)
    )

    return loss

# Analytical gradients using backpropagation
def analytical_gradients(X, y, W1, b1, W2, b2):

    m = len(X)

    A1, A2 = forward(X, W1, b1, W2, b2)

    dZ2 = (A2 - y) / m

    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (1 - A1**2)

    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    return dW1, db1, dW2, db2

# Numerical gradient using central differences
def numerical_gradient(X, y, W1, b1, W2, b2, parameter):

    epsilon = 1e-5

    gradient = np.zeros_like(parameter)

    for index in np.ndindex(parameter.shape):

        original = parameter[index]

        parameter[index] = original + epsilon

        loss_plus = calculate_loss(X, y, W1, b1, W2, b2)

        parameter[index] = original - epsilon

        loss_minus = calculate_loss(X, y, W1, b1, W2, b2)

        parameter[index] = original

        gradient[index] = (
            loss_plus - loss_minus
        ) / (2 * epsilon)

    return gradient

# Calculate analytical gradients
dW1, db1, dW2, db2 = analytical_gradients(
    X, y, W1, b1, W2, b2
)

# Calculate numerical gradients
nW1 = numerical_gradient(X, y, W1, b1, W2, b2, W1)
nb1 = numerical_gradient(X, y, W1, b1, W2, b2, b1)
nW2 = numerical_gradient(X, y, W1, b1, W2, b2, W2)
nb2 = numerical_gradient(X, y, W1, b1, W2, b2, b2)

# Compare gradients
def compare_gradients(analytical, numerical):

    difference = np.linalg.norm(
        analytical - numerical
    )

    denominator = (
        np.linalg.norm(analytical) +
        np.linalg.norm(numerical) + 1e-12
    )

    relative_error = difference / denominator

    return relative_error

print("GRADIENT CHECKING RESULTS")
print("-------------------------")

print("W1 Error:", compare_gradients(dW1, nW1))
print("b1 Error:", compare_gradients(db1, nb1))
print("W2 Error:", compare_gradients(dW2, nW2))
print("b2 Error:", compare_gradients(db2, nb2))

total_error = compare_gradients(
    np.concatenate([
        dW1.flatten(), db1.flatten(),
        dW2.flatten(), db2.flatten()
    ]),
    np.concatenate([
        nW1.flatten(), nb1.flatten(),
        nW2.flatten(), nb2.flatten()
    ])
)

print("\nOverall Relative Error:", total_error)

if total_error < 1e-5:
    print("Gradient checking PASSED!")
else:
    print("Gradient checking FAILED!")

GRADIENT CHECKING RESULTS
-------------------------
W1 Error: 1.0030191672250267e-08
b1 Error: 9.893139302628518e-09
W2 Error: 1.0027258348891115e-08
b2 Error: 9.8013546063613e-09

Overall Relative Error: 9.81452725533927e-09
Gradient checking PASSED!


##Observations and Conclusion

1. Analytical gradients are calculated using backpropagation.
2. Numerical gradients are calculated using finite differences.
3. Both methods produce nearly identical gradient values.
4. The relative error is very small.
5. This confirms that the backpropagation implementation is mathematically correct.

Conclusion

Gradient checking is an important technique for debugging neural networks. It verifies the correctness of analytical gradients by comparing them with numerical approximations.